# Wikipedia Verisiyle GPT2 Fine-tuning

## 1. Kurulum

In [ ]:
!pip install -q requests beautifulsoup4 transformers datasets accelerate

## 2. Wikipedia'dan Web Scraping

Aşağıdaki hücre, belirttiğiniz Wikipedia kategorisindeki makaleleri
gerçek HTML scraping (requests + BeautifulSoup) ile çeker.


In [ ]:
import requests
from bs4 import BeautifulSoup
import time, re, os, json

HEADERS = {
    "User-Agent": "WikiScraperAcademicProject/1.0 (educational use; contact: your_email@example.com)"
}
BASE_URL = "https://en.wikipedia.org/wiki/"
API_URL = "https://en.wikipedia.org/w/api.php"
STOP_SECTIONS = {"references", "see also", "external links", "notes",
                  "further reading", "bibliography", "citations"}


def get_category_members(category, limit=50):
    members = []
    cmcontinue = None
    while len(members) < limit:
        params = {
            "action": "query", "list": "categorymembers",
            "cmtitle": f"Category:{category}",
            "cmlimit": min(500, limit - len(members)), "format": "json",
        }
        if cmcontinue:
            params["cmcontinue"] = cmcontinue
        resp = requests.get(API_URL, params=params, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        data = resp.json()
        for item in data.get("query", {}).get("categorymembers", []):
            if item["ns"] == 0:
                members.append(item["title"])
        cmcontinue = data.get("continue", {}).get("cmcontinue")
        if not cmcontinue:
            break
    return members[:limit]


def clean_text(text):
    text = re.sub(r"\[\d+\]", "", text)
    text = re.sub(r"\[edit\]", "", text)
    text = re.sub(r"\[citation needed\]", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def scrape_article(title):
    url = BASE_URL + title.replace(" ", "_")
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
    except requests.RequestException as e:
        print(f"  [HATA] {title}: {e}")
        return None
    if resp.status_code != 200:
        print(f"  [HATA] {title}: HTTP {resp.status_code}")
        return None
    soup = BeautifulSoup(resp.text, "html.parser")
    content_div = soup.find("div", {"class": "mw-parser-output"})
    if content_div is None:
        return None
    paragraphs = []
    for tag in content_div.find_all(["p", "h2"], recursive=True):
        if tag.name == "h2":
            heading = tag.get_text(strip=True).lower()
            if any(stop in heading for stop in STOP_SECTIONS):
                break
            continue
        text = tag.get_text(" ", strip=True)
        if text:
            paragraphs.append(text)
    full_text = clean_text(" ".join(paragraphs))
    return full_text if len(full_text) > 200 else None


def scrape_titles(titles, delay=0.5):
    results = {}
    for i, title in enumerate(titles, 1):
        print(f"[{i}/{len(titles)}] Scraping: {title}")
        text = scrape_article(title)
        if text:
            results[title] = text
        time.sleep(delay)
    return results


CATEGORY = "Machine learning"  # <-- istediğiniz konuyla değiştirin
LIMIT = 60

titles = get_category_members(CATEGORY, limit=LIMIT)
print(f"'{CATEGORY}' kategorisinde {len(titles)} makale bulundu. Scraping başlıyor...\n")
wiki_data = scrape_titles(titles)

with open("wiki_corpus.json", "w", encoding="utf-8") as f:
    json.dump(wiki_data, f, ensure_ascii=False, indent=2)

total_chars = sum(len(v) for v in wiki_data.values())
print(f"\nTamamlandı. {len(wiki_data)} makale toplandı. Toplam karakter: {total_chars:,}")

'Machine learning' kategorisinde 60 makale bulundu. Scraping başlıyor...

[1/60] Scraping: Machine learning
[2/60] Scraping: Outline of machine learning
[3/60] Scraping: 80 Million Tiny Images
[4/60] Scraping: A Logical Calculus of the Ideas Immanent in Nervous Activity
[5/60] Scraping: Accelerated Linear Algebra
[6/60] Scraping: Action model learning
[7/60] Scraping: Active learning (machine learning)
[8/60] Scraping: Adversarial machine learning
[9/60] Scraping: AI data center
[10/60] Scraping: AIOps
[11/60] Scraping: AIXI
[12/60] Scraping: Algorithm selection
[13/60] Scraping: Algorithmic bias
[14/60] Scraping: Algorithmic inference
[15/60] Scraping: AlphaChip (controversy)
[16/60] Scraping: Anomaly detection
[17/60] Scraping: Aporia (company)
[18/60] Scraping: Apprenticeship learning
[19/60] Scraping: Artificial intelligence in hiring
[20/60] Scraping: Astrostatistics
[21/60] Scraping: Attention (machine learning)
[22/60] Scraping: Audio inpainting
[23/60] Scraping: Automated decis

## 3. Veriyi train/val olarak ayır

In [ ]:
import random
random.seed(42)

MIN_ARTICLE_CHARS = 300
TRAIN_RATIO = 0.9

articles = [t for t in wiki_data.values() if len(t) >= MIN_ARTICLE_CHARS]
random.shuffle(articles)

split_idx = int(len(articles) * TRAIN_RATIO)
train_articles = articles[:split_idx]
val_articles = articles[split_idx:]

def save_corpus(articles, path):
    with open(path, "w", encoding="utf-8") as f:
        for art in articles:
            f.write(art.strip() + "\n\n")

save_corpus(train_articles, "train.txt")
save_corpus(val_articles, "val.txt")

print(f"Train: {len(train_articles)} makale | Val: {len(val_articles)} makale")
print("train.txt ve val.txt oluşturuldu.")

Train: 53 makale | Val: 6 makale
train.txt ve val.txt oluşturuldu.


## 4. Tokenizer ve modeli yükle

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "gpt2"  # GPT-2 (~124M parametre)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 ailesinde pad token yok, eos ile dolduruyoruz

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
print(f"Model parametre sayısı: {model.num_parameters():,}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model parametre sayısı: 124,439,808


## 5. Veriyi tokenize et

Metinleri `block_size` uzunluğunda bloklara bölüyoruz (GPT-2 tarzı causal LM eğitimi için standart yöntem).

In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset(
    "text",
    data_files={"train": "train.txt", "validation": "val.txt"},
    sample_by="paragraph",
)
raw_datasets

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 53
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 6
    })
})

In [ ]:
BLOCK_SIZE = 256

def tokenize_function(examples):
    return tokenizer(examples["text"])

tokenized_datasets = raw_datasets.map(
    tokenize_function, batched=True, remove_columns=["text"]
)

def group_texts(examples):
    concatenated = sum(examples["input_ids"], [])
    total_length = (len(concatenated) // BLOCK_SIZE) * BLOCK_SIZE
    result_ids = [concatenated[i:i + BLOCK_SIZE] for i in range(0, total_length, BLOCK_SIZE)]
    return {"input_ids": result_ids, "labels": [ids[:] for ids in result_ids]}

lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    remove_columns=tokenized_datasets["train"].column_names,  # <-- eklenen satır
)
lm_datasets

Map:   0%|          | 0/53 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2049 > 1024). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Map:   0%|          | 0/53 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 437
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 61
    })
})

## 6. Trainer ile fine-tune et

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./wiki-distilgpt2",
    num_train_epochs=6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["validation"],
    data_collator=data_collator,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.385101,3.366516
2,3.085770,3.353115
3,2.962385,3.361891
4,2.857200,3.362129
5,2.815459,3.375162
6,2.849863,3.375646


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=330, training_loss=2.996206844214237, metrics={'train_runtime': 416.3517, 'train_samples_per_second': 6.298, 'train_steps_per_second': 0.793, 'total_flos': 342553853952000.0, 'train_loss': 2.996206844214237, 'epoch': 6.0})

## 7. Modeli test et

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)

prompt = "Machine learning is"
outputs = generator(
    prompt,
    max_new_tokens=120,       # ~5-6 cümle hedefi
    min_new_tokens=80,        # erken durmasın
    num_return_sequences=2,
    do_sample=True,
    top_p=0.9,
    top_k=50,
    temperature=0.85,
    repetition_penalty=1.3,
    no_repeat_ngram_size=3,
    clean_up_tokenization_spaces=False,
)
for i, out in enumerate(outputs, 1):
    print(f"--- Üretim {i} ---")
    print(out["generated_text"])
    print()

[transformers] Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'top_k', 'no_repeat_ngram_size', 'temperature', 'min_new_tokens', 'top_p', 'num_return_sequences', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=80) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Üretim 1 ---
Machine learning is a field of study where machine-learning methods are used to generate models that can accurately predict the content, actions or outcomes in real life situations. [ 1 ] In AI studies , for example : Researchers have observed how data from websites and social media correlate with their users' preferences (e., "pornography" ) which may result into bias against particular groups . A recent paper by Pierre Crouzet concluded: Data mining tools might help researchers improve understanding human behavior through new approaches such as predictive modeling algorithms designed specifically toward capturing accurate patterns across multiple sources; training them on specific examples based upon historical

--- Üretim 2 ---
Machine learning is a challenging field. Some of the most commonly used methods for building machine-learning models are ML training and supervised prediction, which require more computation power than traditional techniques (e .g., linear re

## 8. Modeli kaydet ve indir

In [ ]:
from google.colab import files

trainer.save_model("./wiki-distilgpt2-final")
tokenizer.save_pretrained("./wiki-distilgpt2-final")

!zip -r wiki-distilgpt2-final.zip wiki-distilgpt2-final
files.download("wiki-distilgpt2-final.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: wiki-distilgpt2-final/ (stored 0%)
  adding: wiki-distilgpt2-final/model.safetensors (deflated 7%)
  adding: wiki-distilgpt2-final/generation_config.json (deflated 24%)
  adding: wiki-distilgpt2-final/config.json (deflated 52%)
  adding: wiki-distilgpt2-final/tokenizer_config.json (deflated 50%)
  adding: wiki-distilgpt2-final/tokenizer.json (deflated 82%)
  adding: wiki-distilgpt2-final/training_args.bin (deflated 53%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>